In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter

# Geographic
import geopandas as gpd
from census import Census
from us import states
import censusdata as acs

In [ ]:
# Define user
user = os.getlogin()

# Working directories
path_sp  = os.path.join('C:\\Users', 'jfontes', 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_git = os.path.join('C:\\Users', 'jfontes', 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

# Set file paths
path_config = os.path.join(path_git, 'Pipeline', 'Python Code', 'Census', 'aa_config')
path_out    = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [ ]:
## Import Variable Mapping
df_vars   = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = 'ACS')
df_inputs = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = 'Inputs'
                        , dtype = {'msa': object})

# Organize inputs into run
indicator_name = df_inputs['indicator_name'].values[0]
year_start = int(df_inputs['year_start'].values[0])
year_end   = int(df_inputs['year_end'  ].values[0])
sp_folder_out = df_inputs['sp_folder'].values[0]

# Subset variables
df_vars = df_vars[df_vars['Indicator Name'] == indicator_name]
df_vars = df_vars[df_vars['Include'] == 'Yes']

# Set tables and variables to import
list_vars = ['NAME'] + df_vars['ID'].to_list()
tables = df_vars['Table'].unique()

# Set years
years_to_import = list(range(year_start, year_end+1))



## Import County FIPS mapping
df_fips = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx')
                        , sheet_name = 'FIPSmapping'
                        , dtype = {'State FIPS': object, 'County FIPS': object})

# Convert to dictionary
dict_fips = df_fips[
                (df_fips['State'].isin(df_inputs['states'].values)) 
                & (df_fips['County Name'].isin(df_inputs['counties'].values))
]
dict_fips = dict_fips[['State FIPS', 'County FIPS']]

dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()


# Set MSA
df_inputs['msa'] = df_inputs['msa'].astype("string")
msa_to_import = df_inputs['msa'].values
msa_to_import = ",".join(msa_to_import)



# view
print(dict_fips)
print(msa_to_import)
print(tables)
print(indicator_name)
print(year_start)
print(year_end)
df_vars.head()

In [ ]:
year = 2022
df_table = df_vars[df_vars['Table'] == tables[0]]

list_table_vars = ['NAME'] + df_table['ID'].to_list() # try with only one variable to speed things up maybe?
variables = ",".join(list_table_vars)

temp = acs5_msa(api_Key     = api_key
                             , variables = variables
                             , year      = year
                             , msa       = msa_to_import)

temp

In [ ]:
# temp.to_excel(os.path.join(path_out, sp_folder_out, 'MSA ID Mapping.xlsx'))